In [1]:
import pandas as pd
import numpy as np
import os

print("Iniciando Script 2: Limpieza, Rangos Dinámicos y Separación de Tablas...")

# 1. CARGA DEL ARCHIVO CRUDO
df = pd.read_parquet("data/raw/sima_raw.parquet")
cols_medicion = ['CO', 'NO', 'NO2', 'NOX', 'O3', 'PM10', 'PM2.5', 'PRS', 'RAINF', 'RH', 'SO2', 'SR', 'TOUT', 'WSR', 'WDR']

# 2. DESTRUCCIÓN DE BANDERAS
print("Eliminando banderas de invalidación y ajustando tipos...")
for col in cols_medicion:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

df['Year'] = df['Date'].dt.year

# 3. FILTRO DE RANGOS FÍSICOS (Según PDF del SIMA)
print("Aplicando rangos operativos del fabricante y del SIMA por año...")
# Diccionario con los límites permitidos (min, max) año por año
rangos = {
    2020: {'PM10': (0, 800), 'PM2.5': (0, 205.94), 'O3': (0, 153), 'WSR': (0, 75), 'TOUT': (0, 41), 'PRS': (690, 750), 'CO': (0, 20), 'NO': (0, 500), 'NO2': (0, 200), 'NOX': (0, 500), 'SO2': (0, 200), 'SR': (0, 1)},
    2021: {'PM10': (0, 800), 'PM2.5': (0, 325), 'O3': (0, 175), 'WSR': (0, 40), 'TOUT': (-6.5, 45), 'PRS': (690, 740), 'CO': (0, 10), 'NO': (0, 350), 'NO2': (0, 100), 'NOX': (0, 400), 'SO2': (0, 300), 'SR': (0, 1)},
    2022: {'PM10': (0, 999), 'PM2.5': (0, 450), 'O3': (0, 160), 'WSR': (0, 35), 'TOUT': (-5, 45), 'PRS': (700, 740), 'CO': (0, 8), 'NO': (0, 400), 'NO2': (0, 175), 'NOX': (0, 420), 'SO2': (0, 200), 'SR': (0, 1.25)},
    2023: {'PM10': (0, 900), 'PM2.5': (0, 800), 'O3': (0, 175), 'WSR': (0, 40), 'TOUT': (0, 45), 'PRS': (690, 740), 'CO': (0, 14), 'NO': (0, 500), 'NO2': (0, 175), 'NOX': (0, 500), 'SO2': (0, 250), 'SR': (0, 1)},
    2024: {'PM10': (0, 999), 'PM2.5': (0, 999), 'O3': (0, 180), 'WSR': (0, 38), 'TOUT': (-4, 45.5), 'PRS': (687.5, 740), 'CO': (0, 18), 'NO': (0, 400), 'NO2': (0, 130), 'NOX': (0, 500), 'SO2': (0, 150), 'SR': (0, 1.26)},
    2025: {'PM10': (0, 820), 'PM2.5': (0, 350), 'O3': (0, 185), 'WSR': (0, 40), 'TOUT': (-4.5, 45), 'PRS': (688, 740), 'CO': (0, 10), 'NO': (0, 350), 'NO2': (0, 175), 'NOX': (0, 400), 'SO2': (0, 405), 'SR': (0, 1.2)}
}

# Límites universales para las que no cambian en el PDF
limites_universales = {'RH': (0, 100), 'WDR': (0, 360)}

# Aplicar filtros
for year, limites in rangos.items():
    mask_year = df['Year'] == year
    for col, (min_val, max_val) in limites.items():
        if col in df.columns:
            # Excepción especial del PDF: Omitir máximo de O3 en NTE2 en 2020
            if year == 2020 and col == 'O3':
                mask_nte2 = mask_year & (df['Estacion'] == 'NTE2') & (df[col] > 153)
                df.loc[mask_nte2, col] = np.nan
            
            # Anular lo que sea físicamente imposible
            fuera_de_rango = mask_year & ((df[col] < min_val) | (df[col] > max_val))
            df.loc[fuera_de_rango, col] = np.nan

# Aplicar universales
for col, (min_val, max_val) in limites_universales.items():
    fuera_de_rango = (df[col] < min_val) | (df[col] > max_val)
    df.loc[fuera_de_rango, col] = np.nan

# 4. SEPARACIÓN EN TABLAS INDIVIDUALES Y EXPORTACIÓN
print("\nSeparando y guardando bases de datos independientes...")
os.makedirs("data/processed/variables", exist_ok=True)

# Llevaremos un registro de cuántas filas válidas sobreviven por variable
resumen_filas = {}

for col in cols_medicion:
    # Nos quedamos con Date, Estacion y la variable específica
    df_var = df[['Date', 'Estacion', col]].copy()
    
    # Eliminamos las filas donde esta variable es NaN (datos limpios y reales 100%)
    df_var_limpia = df_var.dropna(subset=[col]).reset_index(drop=True)
    
    # Exportamos a Parquet
    ruta_salida = f"data/processed/variables/{col}_clean.parquet"
    df_var_limpia.to_parquet(ruta_salida, index=False)
    
    resumen_filas[col] = df_var_limpia.shape[0]
    print(f"✔️ {col} guardada. Filas válidas: {df_var_limpia.shape[0]:,}")

print("\n✅ ¡Fase 2 Completada! Tus variables están limpias, validadas y listas en 'data/processed/variables/'")

Iniciando Script 2: Limpieza, Rangos Dinámicos y Separación de Tablas...
Eliminando banderas de invalidación y ajustando tipos...
Aplicando rangos operativos del fabricante y del SIMA por año...

Separando y guardando bases de datos independientes...
✔️ CO guardada. Filas válidas: 654,137
✔️ NO guardada. Filas válidas: 627,938
✔️ NO2 guardada. Filas válidas: 623,499
✔️ NOX guardada. Filas válidas: 627,434
✔️ O3 guardada. Filas válidas: 643,436
✔️ PM10 guardada. Filas válidas: 719,672
✔️ PM2.5 guardada. Filas válidas: 564,250
✔️ PRS guardada. Filas válidas: 708,099
✔️ RAINF guardada. Filas válidas: 718,843
✔️ RH guardada. Filas válidas: 668,445
✔️ SO2 guardada. Filas válidas: 641,155
✔️ SR guardada. Filas válidas: 725,814
✔️ TOUT guardada. Filas válidas: 701,649
✔️ WSR guardada. Filas válidas: 701,033
✔️ WDR guardada. Filas válidas: 691,581

✅ ¡Fase 2 Completada! Tus variables están limpias, validadas y listas en 'data/processed/variables/'
